# Comparación de Planes de Pago de Préstamos Estudiantiles con PROC LOAN

## Resumen Ejecutivo

Una oficina de ayuda financiera evalúa cómo debería pagar una cohorte de graduados un saldo representativo de $27,500 comparando cuatro estructuras de pago -- un plan federal estándar de tasa fija, un refinanciamiento privado con comisión de apertura, un préstamo de tasa variable (ARM) con tope, y una reducción de tasa patrocinada por el empleador -- usando `PROC LOAN`. En un plazo de 120 meses, las cuatro ofertas se alinean claramente en pago mensual e interés total a sus tasas iniciales cotizadas: el plan federal estándar cuesta más en interés (**$10,021.22** al 6.53%, pago **$312.68**), el refinanciamiento privado reduce la tasa pero añade una comisión de $412.50 (**$9,120.20** al 5.99%, pago **$305.17**), y el ARM con menor tasa cotizada (**$7,099.76** al 4.75%, pago **$288.33**) y la reducción del empleador (**$6,700.67** al 4.50%, pago **$285.01**) llevan las facturas de interés más bajas. Un bloque `COMPARE` luego reporta el interés acumulado, principal y saldo pendiente de cada plan a los 36, 60 y 120 meses, mostrando cuánto se ha amortizado cada préstamo en los puntos en los que un prestatario tiene más probabilidad de refinanciar o liquidar.

## Fuentes de Datos

| Conjunto de datos | Filas | Descripción | Variables clave |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Perfiles sintéticos de préstamos de una cohorte de graduados, generados directamente con `call streaminit(20260531)` y `rand('uniform')`. Se usan para motivar términos de préstamo realistas para la comparación. | `student_id` (1001-1040), `balance` (principal al graduarse; este muestreo abarca $11,800-$47,750, media $30,771), `apr` (tasa nominal anual 4.10%-9.10%, media 6.50%), `term` (120 o 180 meses, media 144), `origination_pct` (comisión 1.0%-2.0%, media 1.50%) |

El prestatario representativo analizado con `PROC LOAN` (monto $27,500, plazo de 120 meses, inicio julio 2026) se ubica cerca del punto medio-bajo de la distribución de saldo y tasa de esta cohorte; no se usan datos externos ni de red. La cohorte existe para motivar términos de préstamo plausibles -- la comparación directa se ejecuta sobre el préstamo representativo único.

# Comparación de Planes de Pago de Préstamos Estudiantiles con PROC LOAN

Cuando los estudiantes se gradúan, una oficina de ayuda financiera debe ayudarlos a elegir entre ofertas de pago que compiten entre sí. `PROC LOAN` (SAS/ETS) está diseñado exactamente para esto: amortiza préstamos de tasa fija, tasa ajustable (ARM) y con reducción de tasa (buydown), y luego los compara lado a lado en pago, interés total y progreso de amortización.

En este cuaderno:

1. Generamos una cohorte sintética de graduados para establecer términos de préstamo realistas.
2. Resumimos la cohorte con `PROC MEANS`.
3. Construimos cuatro planes de pago para un saldo representativo de $27,500 y los comparamos con `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Volvemos a ejecutar el plan fijo recomendado por sí solo para confirmar su economía independiente.

Todo se ejecuta localmente sobre datos sintéticos generados directamente -- sin archivos externos ni acceso a la red.

## Paso 1 — Generar una cohorte sintética de graduados

Simulamos 40 prestatarios. Cada uno obtiene un saldo de principal al graduarse, una TAE ligada holgadamente a la calidad crediticia, un plazo de pago estándar (10 o 15 años) y un porcentaje de comisión de apertura. La semilla hace los datos reproducibles.

In [1]:
DATOS borrowers;
   LLAMAR streaminit(20260531);
   LONGITUD plan $ 28;
   HACER student_id = 1001 HASTA 1040;
      /* Saldo de principal al graduarse: $9,500 - $48,000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* TAE nominal anual por nivel crediticio: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Plazo de pago estandar: 120 o 180 meses */
      SI rand('uniform') < 0.6 ENTONCES term = 120;
      SINO term = 180;
      /* Comision de apertura como porcentaje del principal: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      SALIDA;
   END;
EJECUTAR;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Paso 2 — Perfilar la cohorte

Antes de modelar préstamos individuales, observamos la distribución de saldos, tasas y plazos. Esto nos dice cómo luce un prestatario *representativo* -- la base para la comparación directa que sigue.

In [2]:
PROCEDIMIENTO MEDIAS DATOS=borrowers n mean MIN MAX maxdec=2;
   ETIQUETA balance='Saldo ($)' apr='TAE (%)' term='Plazo (meses)' origination_pct='Comision de Apertura (%)';
   VAR balance apr term origination_pct;
EJECUTAR;

                                                  The MEANS Procedure

 Variable         Label                           N           Mean     Minimum     Maximum
 -----------------------------------------------------------------------------------------
 balance          Saldo ($)                      40       30771.25    11800.00    47750.00
 apr              TAE (%)                        40           6.50        4.10        9.10
 term             Plazo (meses)                  40         144.00      120.00      180.00
 origination_pct  Comision de Apertura (%)       40           1.50        1.00        2.00
 -----------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Paso 3 — Comparar cuatro planes de pago para un prestatario representativo

Usando un saldo representativo de $27,500 en un plazo de 120 meses (10 años) que inicia en julio de 2026, alineamos cuatro ofertas realistas:

- **Federal Directo No Subsidiado (Estándar)** — un préstamo simple de tasa fija al 6.53%.
- **Refinanciamiento Privado (con comisión)** — una tasa fija más baja del 5.99%, pero con un costo de inicialización de $412.50 (`INIT=`).
- **ARM Variable Privado** — un préstamo ajustable al 4.75% que lleva un tope de 1% por ajuste / 5% de por vida (`CAPS=`), un `MAXRATE=` del 9.75%, `ADJUSTFREQ=` anual, y la palabra clave `WORSTCASE`.
- **Reducción del Empleador 2-1** — un inicio subsidiado del 4.50% que, según el calendario cotizado, sube mediante `BDRATES=` a 5.50% en el año 2 y a 6.50% después.

La sentencia `COMPARE` solicita la vista cruzada entre préstamos a los 36, 60 y 120 meses, con un `TAXRATE=` del 22% y una tasa mínima atractiva de retorno del 4% (`MARR=`); `OUTSUM=` y `OUTCOMP=` capturan el resumen por préstamo y las filas de comparación. El listado a continuación reporta, para cada plan y cada punto de control, el **interés acumulado pagado, el principal acumulado amortizado y el saldo pendiente** -- una vista clara del progreso de amortización entre los candidatos.

> **Nota sobre los ajustes de tasa.** `PROC LOAN` de Jenner amortiza cada plan a su tasa **inicial** cotizada, por lo que los ajustes `CAPS`/`MAXRATE`/`WORSTCASE` del ARM y los pasos `BDRATES` de la reducción del empleador se muestran en el listado como los términos del préstamo, pero **no** se aplican a la matemática del pago -- las cifras del ARM y de la reducción a continuación reflejan sus tasas iniciales de 4.75% y 4.50% mantenidas fijas durante todo el plazo. Trate esos dos totales como costos en el mejor de los casos (tasa inicial), no como techos del peor caso.

In [3]:
PROCEDIMIENTO loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label='Federal Directo No Subsidiado (Estandar)';

   fixed rate=5.99 init=412.50
         label='Refinanciamiento Privado (con comision)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label='ARM Variable Privado (peor caso)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label='Reduccion del Empleador 2-1, luego 6.5%';

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
EJECUTAR;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Federal Directo No Subsidiado (Estandar)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Refinanciamiento Privado (con comision)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: ARM Variable Privado (peor caso)
    Loan Type:                   Adjustable Rate
    Amount (Princ


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Paso 4 — Volver a ejecutar el plan fijo recomendado por sí solo

Para el prestatario que valora la certeza del pago, el plan federal estándar de tasa fija es la opción segura por defecto. Lo volvemos a ejecutar de forma aislada para confirmar su economía principal: el mismo pago mensual de **$312.68**, **$37,521.22** de total pagado y **$10,021.22** de interés total visto en la comparación de las cuatro opciones, ahora presentado como un resumen de préstamo independiente.

In [4]:
PROCEDIMIENTO loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label='Federal Directo No Subsidiado (Estandar)';
EJECUTAR;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Federal Directo No Subsidiado (Estandar)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Federal Directo No Subsidiado (Estandar)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interpretación de los resultados

Los cuatro planes amortizan completamente el mismo principal de $27,500 en 120 meses (el saldo pendiente de cada plan llega a $0.00 en el mes 120 en la tabla COMPARE), así que la decisión gira en torno al **pago mensual** y al **interés total a la tasa cotizada**:

| Plan | Tasa | Pago | Interés total | Notas |
|------|------|---------|----------------|-------|
| Federal Directo Estándar | 6.53% | $312.68 | $10,021.22 | Sin comisión; protecciones más sólidas para el prestatario |
| Refinanciamiento Privado | 5.99% | $305.17 | $9,120.20 | Comisión de apertura de $412.50 |
| ARM Variable Privado | 4.75% (inicial) | $288.33 | $7,099.76 | La tasa puede subir; la cifra es el costo a la tasa inicial |
| Reducción del Empleador 2-1 | 4.50% (inicial) | $285.01 | $6,700.67 | Depende de la continuidad del empleo |

- El plan **federal estándar** es el más costoso en interés ($10,021.22) pero ofrece un pago fijo y predecible de $312.68 sin comisión.
- El **refinanciamiento privado** reduce la tasa al 5.99% ($9,120.20 de interés, $901 menos que el plan federal) pero cobra una comisión de apertura de $412.50, así que su ventaja neta sobre el plan federal es aproximadamente $488 de interés menos la comisión -- significativa solo si el préstamo dura lo suficiente como para no ser refinanciado antes.
- El **ARM** y la **reducción** muestran el interés más bajo aquí ($7,099.76 y $6,700.67) puramente porque sus tasas **iniciales** están muy por debajo de las ofertas fijas. Como se señaló en el Paso 3, Jenner mantiene esas tasas iniciales fijas, así que estas son cifras del mejor de los casos: un ARM real que se ajusta al alza -- o una reducción que pierde su subsidio del empleador -- resultaría más alto, y el pago no está garantizado.

La tabla `COMPARE` afina esto mostrando qué tan rápido amortiza cada plan. A los 36 meses, el plan federal ha pagado $4,792.27 de interés y amortizado $6,464.10 de principal (saldo $21,035.90), mientras que la reducción del empleador ha pagado solo $3,263.97 de interés y amortizado $6,996.24 de principal (saldo $20,503.76) -- los planes de menor tasa cuestan menos interés y reducen el principal más rápido durante los primeros tres años. Para un graduado averso al riesgo, la certeza de tasa del plan federal estándar suele justificar su mayor interés; para un prestatario confiado en un empleo estable y duradero, el inicio subsidiado de la reducción del empleador entrega el mayor ahorro entre las opciones que realmente fijan su tasa.